In [11]:
import numpy as np
import cv2
from scipy.signal import fftconvolve
import torch
import torch.nn.functional as F
import math

In [12]:
SQRT_CLIP = math.sqrt(1500.0)

BAND_MEAN = np.array(
    [15.0381, 14.5305, 14.4030, 15.4191, 13.6231, 14.2143, 14.7041, 13.1745],
    dtype=np.float32,
)

BAND_STD = np.array(
    [8.2196, 10.6197, 9.4811, 9.0923, 10.5712, 10.4277, 10.3784, 9.7216],
    dtype=np.float32,
)

In [18]:
def normalize_phisat2_batch(x, selected_band_indices=None):
    """
    x: shape (N,8,H,W) or (8,H,W), raw PhiSat-2 domain
    """
    x = np.asarray(x, dtype=np.float32)

    single = x.ndim == 3
    if single:
        x = x[None]

    if selected_band_indices is None:
        selected_band_indices = np.arange(8)

    x = np.sqrt(np.clip(x, 0.0, None))
    x = np.clip(x, 0.0, SQRT_CLIP)

    x = x[:, selected_band_indices]

    mean = BAND_MEAN[selected_band_indices][None, :, None, None]
    std = BAND_STD[selected_band_indices][None, :, None, None]

    x = (x - mean) / (std + 1e-6)

    return x[0] if single else x

In [3]:
def apply_gaussian_blur_multiband(x, severity=5):
    """
    x: np.ndarray, shape (C,H,W) or (N,C,H,W)
       e.g. (7,128,128), (8,256,256), or batches.
    Returns float32 with same shape.
    """
    kernel_sizes = [3, 5, 7, 9, 11]
    kernel_size = kernel_sizes[severity - 1]

    x = np.asarray(x, dtype=np.float32)
    single = x.ndim == 3
    if single:
        x = x[None]

    out = np.empty_like(x, dtype=np.float32)

    for n in range(x.shape[0]):
        for c in range(x.shape[1]):
            out[n, c] = cv2.GaussianBlur(
                x[n, c],
                (kernel_size, kernel_size),
                sigmaX=0,
                borderType=cv2.BORDER_REFLECT_101,
            )

    return out[0] if single else out

In [4]:
def precompute_k_grid(height, width):
    fy = np.fft.fftfreq(height)
    fx = np.fft.fftfreq(width)
    FX, FY = np.meshgrid(fx, fy)
    k = np.sqrt(FX**2 + FY**2) / 0.5
    return np.clip(k, 0.0, 1.0).astype(np.float32)


def mtf_to_psf(mtf_filter):
    psf = np.real(np.fft.ifft2(mtf_filter))
    psf = np.fft.fftshift(psf)
    psf = np.maximum(psf, 0.0)
    psf = psf / (psf.sum() + 1e-8)
    return psf.astype(np.float32)


def perturb_mtf_scalar(x, mtf_nyquist, k_grid=None):
    """
    x: shape (C,H,W) or (N,C,H,W)
    mtf_nyquist: scalar MTF@Nyquist
    """
    x = np.asarray(x, dtype=np.float32)
    single = x.ndim == 3
    if single:
        x = x[None]

    n, c, h, w = x.shape
    if k_grid is None:
        k_grid = precompute_k_grid(h, w)

    mtf_filter = np.exp(np.log(float(mtf_nyquist)) * (k_grid ** 2)).astype(np.float32)
    psf = mtf_to_psf(mtf_filter)

    out = np.empty_like(x, dtype=np.float32)
    for i in range(n):
        for b in range(c):
            out[i, b] = fftconvolve(x[i, b], psf, mode="same").astype(np.float32)

    return out[0] if single else out

In [13]:
def make_mmd_features_from_normalized(x, out_hw=32, max_samples=512):
    """
    x: normalized batch, shape (N,C,H,W)
    """
    x = torch.from_numpy(np.asarray(x, dtype=np.float32))

    x = F.interpolate(x, size=(out_hw, out_hw), mode="area")
    x = x.flatten(start_dim=1)

    if x.shape[0] > max_samples:
        idx = torch.randperm(x.shape[0])[:max_samples]
        x = x[idx]

    return x

In [16]:
def calibrate_phisat2_mtf_to_gaussian_normalized(
    image_batch,
    gaussian_severity=5,
    candidate_mtf_values=None,
    selected_band_indices=None,
    feature_hw=32,
):
    if candidate_mtf_values is None:
        candidate_mtf_values = np.array([
            0.50, 0.40, 0.30, 0.225, 0.18, 0.15,
            0.12, 0.10, 0.08, 0.072, 0.06, 0.0555,
            0.05, 0.04, 0.039, 0.03, 0.02, 0.01
        ], dtype=np.float32)

    image_batch = np.asarray(image_batch, dtype=np.float32)

    # Target: raw -> Gaussian blur -> dataset normalization
    gaussian_raw = apply_gaussian_blur_multiband(
        image_batch,
        severity=gaussian_severity,
    )

    gaussian_norm = normalize_phisat2_batch(
        gaussian_raw,
        selected_band_indices=selected_band_indices,
    )

    target_feat = make_mmd_features_from_normalized(
        gaussian_norm,
        out_hw=feature_hw,
    )

    h, w = image_batch.shape[-2:]
    k_grid = precompute_k_grid(h, w)

    results = []

    for mtf in candidate_mtf_values:
        # Candidate: raw -> MTF blur -> dataset normalization
        mtf_raw = perturb_mtf_scalar(
            image_batch,
            mtf_nyquist=float(mtf),
            k_grid=k_grid,
        )

        mtf_norm = normalize_phisat2_batch(
            mtf_raw,
            selected_band_indices=selected_band_indices,
        )

        mtf_feat = make_mmd_features_from_normalized(
            mtf_norm,
            out_hw=feature_hw,
        )

        score = rbf_mmd2(target_feat, mtf_feat)

        results.append({
            "mtf_nyquist": float(mtf),
            "mmd2": score,
        })

    return sorted(results, key=lambda d: d["mmd2"])

In [5]:
def rbf_mmd2(x, y, sigmas=(1, 2, 4, 8, 16)):
    """
    x, y: torch tensors, shape (N,D)
    Returns squared MMD with multi-scale RBF kernels.
    """
    x = x.float()
    y = y.float()

    xx = torch.cdist(x, x) ** 2
    yy = torch.cdist(y, y) ** 2
    xy = torch.cdist(x, y) ** 2

    mmd = 0.0
    for s in sigmas:
        gamma = 1.0 / (2 * s ** 2)
        mmd += torch.exp(-gamma * xx).mean()
        mmd += torch.exp(-gamma * yy).mean()
        mmd -= 2 * torch.exp(-gamma * xy).mean()

    return mmd.item()

In [6]:
def make_mmd_features(x, out_hw=32, max_samples=512):
    """
    x: np.ndarray, shape (N,C,H,W)
    Downsamples spatially and flattens for MMD.
    """
    x = torch.from_numpy(np.asarray(x, dtype=np.float32))

    if x.ndim != 4:
        raise ValueError(f"Expected shape (N,C,H,W), got {tuple(x.shape)}")

    x = F.interpolate(x, size=(out_hw, out_hw), mode="area")
    x = x.flatten(start_dim=1)

    if x.shape[0] > max_samples:
        idx = torch.randperm(x.shape[0])[:max_samples]
        x = x[idx]

    # Normalize feature scale for stable RBF bandwidths
    x = (x - x.mean(dim=0, keepdim=True)) / (x.std(dim=0, keepdim=True) + 1e-6)

    return x

In [7]:
def calibrate_mtf_to_reobench_gaussian(
    image_batch,
    gaussian_severity=5,
    candidate_mtf_values=None,
    feature_hw=32,
):
    """
    Finds scalar MTF@Nyquist whose blur distribution is closest to
    REOBench-style Gaussian severity using MMD.

    image_batch: np.ndarray, shape (N,C,H,W)
    """
    if candidate_mtf_values is None:
        candidate_mtf_values = np.array([
            0.50, 0.40, 0.30, 0.225, 0.18, 0.15,
            0.12, 0.10, 0.08, 0.0555, 0.04, 0.03, 0.02
        ], dtype=np.float32)

    image_batch = np.asarray(image_batch, dtype=np.float32)

    target = apply_gaussian_blur_multiband(
        image_batch,
        severity=gaussian_severity,
    )

    target_feat = make_mmd_features(target, out_hw=feature_hw)

    h, w = image_batch.shape[-2:]
    k_grid = precompute_k_grid(h, w)

    results = []

    for mtf in candidate_mtf_values:
        mtf_blurred = perturb_mtf_scalar(
            image_batch,
            mtf_nyquist=float(mtf),
            k_grid=k_grid,
        )

        mtf_feat = make_mmd_features(mtf_blurred, out_hw=feature_hw)
        score = rbf_mmd2(target_feat, mtf_feat)

        results.append({
            "mtf_nyquist": float(mtf),
            "mmd2": score,
        })

    results = sorted(results, key=lambda d: d["mmd2"])
    return results

In [8]:
from pathlib import Path
import numpy as np

DATASET_ROOT = Path("~/scratch/segmentation_dataset_v1").expanduser()

S2_DIR = DATASET_ROOT / "images_s2_npy"
PHISAT2_DIR = DATASET_ROOT / "images_phisat2_npy"


def load_random_npy_batch(folder, batch_size=64, seed=0):
    """
    Loads random .npy images from folder.

    Expected per-file shapes:
      (7,128,128), (8,256,256), or possibly (H,W,C)

    Returns:
      batch: np.ndarray, shape (N,C,H,W), float32
      files: selected file paths
    """
    folder = Path(folder)
    files = sorted(folder.glob("*.npy"))

    if len(files) == 0:
        raise FileNotFoundError(f"No .npy files found in {folder}")

    rng = np.random.default_rng(seed)
    chosen = rng.choice(files, size=min(batch_size, len(files)), replace=False)

    batch = []
    for f in chosen:
        x = np.load(f).astype(np.float32)

        # Convert H,W,C to C,H,W if needed
        if x.ndim == 3 and x.shape[-1] in (7, 8):
            x = np.moveaxis(x, -1, 0)

        if x.ndim != 3:
            raise ValueError(f"{f} has invalid shape {x.shape}")

        batch.append(x)

    batch = np.stack(batch, axis=0).astype(np.float32)
    return batch, chosen

In [9]:
s2_batch, s2_files = load_random_npy_batch(S2_DIR, batch_size=64, seed=42)
phisat2_batch, phisat2_files = load_random_npy_batch(PHISAT2_DIR, batch_size=64, seed=42)

print("S2 batch:", s2_batch.shape, s2_batch.dtype)
print("PhiSat-2 batch:", phisat2_batch.shape, phisat2_batch.dtype)

S2 batch: (64, 7, 128, 128) float32
PhiSat-2 batch: (64, 8, 256, 256) float32


In [19]:
phisat2_results = calibrate_phisat2_mtf_to_gaussian_normalized(
    phisat2_batch,
    gaussian_severity=5,
)

phisat2_results[:10]

[{'mtf_nyquist': 0.009999999776482582, 'mmd2': 0.05767640471458435},
 {'mtf_nyquist': 0.019999999552965164, 'mmd2': 0.058014124631881714},
 {'mtf_nyquist': 0.029999999329447746, 'mmd2': 0.05836459994316101},
 {'mtf_nyquist': 0.039000000804662704, 'mmd2': 0.05866733193397522},
 {'mtf_nyquist': 0.03999999910593033, 'mmd2': 0.05870300531387329},
 {'mtf_nyquist': 0.05000000074505806, 'mmd2': 0.05901908874511719},
 {'mtf_nyquist': 0.05550000071525574, 'mmd2': 0.05918663740158081},
 {'mtf_nyquist': 0.05999999865889549, 'mmd2': 0.05932062864303589},
 {'mtf_nyquist': 0.07199999690055847, 'mmd2': 0.05966317653656006},
 {'mtf_nyquist': 0.07999999821186066, 'mmd2': 0.05986684560775757}]

In [10]:
s2_results = calibrate_mtf_to_reobench_gaussian(
    s2_batch,
    gaussian_severity=5,
)

phisat2_results = calibrate_mtf_to_reobench_gaussian(
    phisat2_batch,
    gaussian_severity=5,
)

print("Best S2:", s2_results[0])
print("Best PhiSat-2:", phisat2_results[0])

Best S2: {'mtf_nyquist': 0.019999999552965164, 'mmd2': 0.11397196352481842}
Best PhiSat-2: {'mtf_nyquist': 0.30000001192092896, 'mmd2': 0.02842634916305542}


In [20]:
# -----------------------------------------------------------------------------
# Calibrate sensor-aware PhiSat-2 SNR noise to match REOBench Gaussian noise
# after PhiSat-2 normalization.
#
# Goal:
#   REOBench target: x_norm + N(0, 0.08)
#
# We find snr_factor such that:
#   normalize(sensor_noise_raw(x_raw, snr_factor)) - normalize(x_raw)
# has std ≈ 0.08.
# -----------------------------------------------------------------------------

import math
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# PhiSat-2 normalization constants
# -----------------------------------------------------------------------------

SQRT_CLIP = math.sqrt(1500.0)

BAND_MEAN = np.array(
    [15.0381, 14.5305, 14.4030, 15.4191, 13.6231, 14.2143, 14.7041, 13.1745],
    dtype=np.float32,
)

BAND_STD = np.array(
    [8.2196, 10.6197, 9.4811, 9.0923, 10.5712, 10.4277, 10.3784, 9.7216],
    dtype=np.float32,
)

PHISAT2_CHANNELS_8 = ["MS1", "MS2", "MS3", "PAN", "MS4", "MS5", "MS6", "MS7"]

# Public PhiSat-2 assumption used in your code:
# MS bands: midpoint of 54--129 = 91.5
# PAN: 256
PHISAT2_SNR_8 = np.array(
    [91.5, 91.5, 91.5, 256.0, 91.5, 91.5, 91.5, 91.5],
    dtype=np.float32,
)


def normalize_phisat2_batch(x, selected_band_indices=None):
    """
    x: raw PhiSat-2 image or batch, shape (8,H,W) or (N,8,H,W)
    returns normalized array with same batch convention.
    """
    x = np.asarray(x, dtype=np.float32)

    single = x.ndim == 3
    if single:
        x = x[None]

    if selected_band_indices is None:
        selected_band_indices = np.arange(8)

    x = np.sqrt(np.clip(x, 0.0, None))
    x = np.clip(x, 0.0, SQRT_CLIP)
    x = x[:, selected_band_indices]

    mean = BAND_MEAN[selected_band_indices][None, :, None, None]
    std = BAND_STD[selected_band_indices][None, :, None, None]

    x = (x - mean) / (std + 1e-6)

    return x[0] if single else x


# -----------------------------------------------------------------------------
# REOBench-style Gaussian noise
# -----------------------------------------------------------------------------

def add_reobench_gaussian_noise_normalized(x_norm, sigma=0.08, seed=None):
    """
    REOBench-style additive Gaussian noise.
    Assumes x_norm is already normalized/model-input space.
    """
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, sigma, size=x_norm.shape).astype(np.float32)
    return (x_norm + noise).astype(np.float32)


# -----------------------------------------------------------------------------
# Sensor-aware SNR noise
# -----------------------------------------------------------------------------

def perturb_snr_phisat2_batch(
    x,
    snr_factor=1.0,
    seed=None,
    use_empirical_lref=True,
):
    """
    Adds raw-space PhiSat-2-like Gaussian noise.

    x: shape (8,H,W) or (N,8,H,W)

    Noise model:
        sigma_raw_b = Lref_b / (SNR_b * snr_factor)

    If use_empirical_lref=True:
        Lref_b = observed batch mean per band, in the same units as x.

    This is recommended for real dataset arrays whose exact radiance units/scale
    may not match the public Lref values.
    """
    x = np.asarray(x, dtype=np.float32)

    single = x.ndim == 3
    if single:
        x_in = x[None]
    else:
        x_in = x

    if x_in.ndim != 4 or x_in.shape[1] != 8:
        raise ValueError(f"Expected shape (8,H,W) or (N,8,H,W), got {x.shape}")

    rng = np.random.default_rng(seed)

    if use_empirical_lref:
        lref_b = x_in.mean(axis=(0, 2, 3)).astype(np.float32)
    else:
        raise NotImplementedError(
            "For this calibration, use_empirical_lref=True is recommended."
        )

    effective_snr_b = PHISAT2_SNR_8 * float(snr_factor)
    sigma_raw_b = lref_b / (effective_snr_b + 1e-12)

    noise = rng.normal(
        loc=0.0,
        scale=sigma_raw_b[None, :, None, None],
        size=x_in.shape,
    ).astype(np.float32)

    out = x_in + noise
    out = np.clip(out, 0.0, None).astype(np.float32)

    return out[0] if single else out


# -----------------------------------------------------------------------------
# Calibration: find snr_factor whose post-normalization noise std ≈ 0.08
# -----------------------------------------------------------------------------

def measure_postnorm_noise_for_snr_factor(
    phisat2_batch,
    snr_factor,
    selected_band_indices=None,
    n_trials=3,
    seed=0,
):
    """
    Measures std of:
        normalize(sensor_noise_raw(x)) - normalize(x)

    Returns global and per-band normalized noise std.
    """
    clean_norm = normalize_phisat2_batch(
        phisat2_batch,
        selected_band_indices=selected_band_indices,
    )

    deltas = []

    for t in range(n_trials):
        noisy_raw = perturb_snr_phisat2_batch(
            phisat2_batch,
            snr_factor=snr_factor,
            seed=seed + t,
            use_empirical_lref=True,
        )

        noisy_norm = normalize_phisat2_batch(
            noisy_raw,
            selected_band_indices=selected_band_indices,
        )

        deltas.append(noisy_norm - clean_norm)

    delta = np.concatenate(deltas, axis=0)

    return {
        "snr_factor": float(snr_factor),
        "effective_snr_ms": float(91.5 * snr_factor),
        "effective_snr_pan": float(256.0 * snr_factor),
        "global_std": float(delta.std()),
        "per_band_std": delta.std(axis=(0, 2, 3)).astype(np.float32),
    }


def calibrate_snr_factor_to_reobench_sigma(
    phisat2_batch,
    target_sigma=0.08,
    selected_band_indices=None,
    candidate_snr_factors=None,
    n_trials=3,
    seed=0,
):
    """
    Finds snr_factor that makes sensor-aware raw noise have post-normalization
    std closest to REOBench sigma.
    """
    if candidate_snr_factors is None:
        candidate_snr_factors = np.array([
            4.0, 3.0, 2.0, 1.5, 1.0,
            0.75, 0.5, 0.35, 0.25,
            0.2, 0.15, 0.1, 0.075, 0.05,
        ], dtype=np.float32)

    rows = []

    for f in candidate_snr_factors:
        stats = measure_postnorm_noise_for_snr_factor(
            phisat2_batch,
            snr_factor=float(f),
            selected_band_indices=selected_band_indices,
            n_trials=n_trials,
            seed=seed,
        )

        rows.append({
            "snr_factor": stats["snr_factor"],
            "effective_snr_ms": stats["effective_snr_ms"],
            "effective_snr_pan": stats["effective_snr_pan"],
            "postnorm_global_std": stats["global_std"],
            "abs_error_to_target": abs(stats["global_std"] - target_sigma),
            "per_band_std": stats["per_band_std"],
        })

    df = pd.DataFrame(rows).sort_values("abs_error_to_target").reset_index(drop=True)
    return df

In [21]:
noise_calib_df = calibrate_snr_factor_to_reobench_sigma(
    phisat2_batch,
    target_sigma=0.08,
    selected_band_indices=None,
    n_trials=5,
    seed=42,
)

noise_calib_df[
    [
        "snr_factor",
        "effective_snr_ms",
        "effective_snr_pan",
        "postnorm_global_std",
        "abs_error_to_target",
    ]
].head(15)

,snr_factor,effective_snr_ms,effective_snr_pan,postnorm_global_std,abs_error_to_target
0,0.250,22.875000,64.000000,0.079309,0.000691
1,0.200,18.300000,51.200001,0.097958,0.017958
2,0.350,32.024999,89.599998,0.057774,0.022226
3,0.500,45.750000,128.000000,0.041570,0.038430
4,0.150,13.725001,38.400002,0.127791,0.047791
5,0.750,68.625000,192.000000,0.028628,0.051372
6,1.000,91.500000,256.000000,0.021879,0.058121
7,1.500,137.250000,384.000000,0.014848,0.065152
8,2.000,183.000000,512.000000,0.011217,0.068783
9,3.000,274.500000,768.000000,0.007528,0.072472


In [ ]:
matched_snr_factor = float(noise_calib_df.iloc[0]["snr_factor"])

x_noisy_raw = perturb_snr_phisat2_batch(
    phisat2_batch,
    snr_factor=matched_snr_factor,
    seed=123,
    use_empirical_lref=True,
)

x_noisy_norm = normalize_phisat2_batch(x_noisy_raw)



[[[[-0.87171    -0.79790777 -0.6478448  ... -0.6159352  -0.6763745
    -0.729316  ]
   [-0.6766416  -0.918342   -0.809738   ... -0.61398435 -0.71583176
    -0.855457  ]
   [-0.792579   -0.76418453 -0.8418214  ... -0.66507685 -0.7181294
    -0.6649285 ]
   ...
   [-1.0379671  -0.7740631  -0.7391033  ... -0.55913717 -0.6523496
    -0.7151574 ]
   [-0.9447486  -0.8844896  -0.90026766 ... -0.68616676 -0.58612597
    -0.6486773 ]
   [-0.9564867  -0.8552361  -0.7726497  ... -0.67096245 -0.52332824
    -0.5922217 ]]

  [[-0.79379976 -0.9059437  -0.90286475 ... -0.7323682  -0.89432293
    -0.6822307 ]
   [-0.8071246  -0.8185685  -0.73214996 ... -0.8606612  -0.7648085
    -0.87499666]
   [-0.7645165  -0.7975418  -0.7133357  ... -0.8598832  -0.7920696
    -0.8759677 ]
   ...
   [-0.75612015 -0.68012017 -0.9042024  ... -0.64524215 -0.7480436
    -0.95741445]
   [-0.7817318  -0.8224765  -0.8308414  ... -0.84092176 -0.85799336
    -0.75208235]
   [-1.0505023  -0.8833232  -0.844467   ... -0.8695637 